# Lander Videos

Elise flies by feel.

This notebook records and displays videos for the preserved 253-score 10D SolarSystemLander checkpoint.

## Set up

In [ ]:
# cell: colab-setup

!git clone https://github.com/miwehle/rl_lab.git
%cd rl_lab
%pip install -r hpo/requirements.txt

import sys

sys.path.insert(0, "dqn/src")
sys.path.insert(0, "hpo/src")

from hpo.notebook.colab import setup_colab

COLAB = setup_colab()

In [ ]:
# cell: video-setup # requires: colab-setup

import json
from pathlib import Path

import torch

from hpo.evaluation.video import record_checkpoint_videos
from hpo.solar_system_lander.environment import EnvFactory, World

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

OBSERVATION_MODE = "10d"
STUDY_SERIES_NAME = f"solar_system_lander_{OBSERVATION_MODE}_elise_accel"
CHECKPOINT_DIR = COLAB.drive_study_dir / "best_checkpoints" / STUDY_SERIES_NAME
CHECKPOINT_PATH = CHECKPOINT_DIR / "best_eval_checkpoint.pt"
CHECKPOINT_METADATA_PATH = CHECKPOINT_DIR / "best_eval_checkpoint.json"
VIDEO_DIR = COLAB.drive_study_dir / "videos" / STUDY_SERIES_NAME

ENV_FACTORY = EnvFactory(OBSERVATION_MODE)

metadata = json.loads(CHECKPOINT_METADATA_PATH.read_text())
print(f"checkpoint: {CHECKPOINT_PATH}")
print(f"score: {metadata['score']:.1f}")
print(f"trial: {metadata['trial_number']}")
metadata["world_scores"]

## Record videos

In [ ]:
WORLDS = [
    World.MERCURY,
    World.VENUS,
    World.EARTH,
    World.MOON,
    World.MARS,
]
SEEDS = [7]  # [7, 42, 1911, 2011, 4711]

In [ ]:
# cell: world-colors; requires: video-setup

from hpo.evaluation.lander_rendering import LanderColors

USE_WORLD_COLORS = True

WORLD_COLORS = {
    World.EARTH: LanderColors(sky=(143, 199, 232), ground=(111, 127, 82), ground_outline=(111, 127, 82)),
    World.VENUS: LanderColors(sky=(214, 168, 92), ground=(138, 103, 64), ground_outline=(138, 103, 64)),
    World.MARS: LanderColors(sky=(201, 149, 122), ground=(139, 79, 61), ground_outline=(139, 79, 61)),
    World.MOON: LanderColors(sky=(5, 7, 10), ground=(119, 122, 125), ground_outline=(119, 122, 125)),
    World.MERCURY: LanderColors(sky=(5, 7, 10), ground=(111, 107, 100), ground_outline=(111, 107, 100)),
}

COLORS_BY_WORLD = [WORLD_COLORS[world] for world in WORLDS] if USE_WORLD_COLORS else None

In [ ]:
# cell: record-videos # requires: video-setup, world-colors

video_paths = record_checkpoint_videos(
    checkpoint_path=CHECKPOINT_PATH,
    environment_factory=ENV_FACTORY,
    worlds=WORLDS,
    seeds=SEEDS,
    output_dir=VIDEO_DIR,
    device=device,
    colors_by_world=COLORS_BY_WORLD,
)

video_paths

## Display world parameters and videos

In [ ]:
# cell: inspect-video-conditions # requires: objective-config, record-videos

from hpo.evaluation.video import display_video, show_video_conditions

show_video_conditions(ENV_FACTORY, WORLDS, SEEDS)


In [ ]:
display_video(video_paths, 7)